# 05 · Comparar tres modelos de embeddings en Qdrant y Chroma

Troceamos el corpus de la documentación de Python 3.15 en chunks de 500 caracteres, calculamos los embeddings con tres modelos (`all-MiniLM-L6-v2`, `bge-base-en-v1.5` y `nomic-embed-text`), los indexamos en **Qdrant** y en **Chroma** (6 colecciones) y evaluamos cada una con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`) y Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`).

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

In [ ]:
import sys
import time
from pathlib import Path

import chromadb
import ollama
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, evidencias_fuera_de_chunks, fuentes_ausentes,
    guardar_jsonl, leer_evidencias, leer_golden_set, leer_jsonl, preparar_documento,
    preparar_query, rango_primer_acierto, rango_primer_acierto_evidencia,
    trocear_corpus,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_JSONL = RAIZ / "data" / "chunks" / "python-3.15-chunks-500.jsonl"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CHROMA = RAIZ / "notebooks" / "chroma_db"
TAMANO_CHUNK = 500
K = 4
LOTE_EMBEDDING = 64   # chunks por llamada al modelo
LOTE_INSERCION = 500  # puntos por inserción (muy por debajo del máximo de Chroma)

## 3. Conexión con Qdrant, Chroma y Ollama

Se comprueba antes de calcular nada, para fallar pronto si falta un servicio. Si Qdrant o Ollama no responden, la celda para con un mensaje que dice qué hacer.


In [ ]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."
    ) from error

chroma = chromadb.PersistentClient(path=str(RUTA_CHROMA))

modelo_ollama = next(cfg["nombre"] for cfg in MODELOS.values() if cfg["proveedor"] == "ollama")
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."
    ) from error

print("Qdrant:", [col.name for col in qdrant.get_collections().collections])
print("Chroma:", [col.name for col in chroma.list_collections()])
print("Ollama:", modelo_ollama, "disponible")

## 4. Troceo del corpus y jsonl

Cada `.md` se parte en tramos consecutivos de 500 caracteres, sin solape. Se guardan en un único jsonl que es la entrada de las seis colecciones. Al volver a ejecutar, el jsonl se regenera.

In [ ]:
chunks = trocear_corpus(CARPETA_CORPUS, TAMANO_CHUNK)
guardar_jsonl(chunks, RUTA_JSONL)
chunks = leer_jsonl(RUTA_JSONL)
print(f"{len(chunks)} chunks guardados en {RUTA_JSONL.relative_to(RAIZ)}")

## 5. Embeddings de los chunks (una vez por modelo)

Los prefijos salen de `MODELOS` (única fuente). Los vectores se calculan una sola vez por modelo y se reutilizan en Qdrant y Chroma. Se mide el tiempo de cada modelo.

In [ ]:
modelos_st = {
    clave: SentenceTransformer(cfg["nombre"])
    for clave, cfg in MODELOS.items() if cfg["proveedor"] == "sentence-transformers"
}


def embeber(clave: str, textos: list[str], barra: bool = False) -> list[list[float]]:
    """Embeddings de una lista de textos ya preparados, en lotes fijos."""
    cfg = MODELOS[clave]
    vectores: list[list[float]] = []
    inicios = range(0, len(textos), LOTE_EMBEDDING)
    for i in tqdm(inicios, desc=clave, disable=not barra):
        lote = textos[i:i + LOTE_EMBEDDING]
        if cfg["proveedor"] == "ollama":
            vectores.extend(ollama.embed(model=cfg["nombre"], input=lote).embeddings)
        else:
            vectores.extend(modelos_st[clave].encode(lote, batch_size=LOTE_EMBEDDING).tolist())
    return vectores


textos_chunks = [c["texto"] for c in chunks]
vectores_chunks: dict[str, list[list[float]]] = {}
t_embeddings: dict[str, float] = {}
for clave in MODELOS:
    docs = [preparar_documento(clave, t) for t in textos_chunks]
    inicio = time.perf_counter()
    vectores_chunks[clave] = embeber(clave, docs, barra=True)
    t_embeddings[clave] = time.perf_counter() - inicio

dimensiones = {clave: len(v[0]) for clave, v in vectores_chunks.items()}
print({clave: f"{t:.1f} s" for clave, t in t_embeddings.items()})
print(dimensiones)

## 6. Seis colecciones con tiempos de inserción

Una colección por modelo en cada base (`pydocs_minilm`, `pydocs_bge`, `pydocs_nomic`), con distancia coseno. Si ya existe, se borra y se vuelve a crear. Cada punto guarda `fuente` y `texto`.

In [ ]:
def nombre_coleccion(clave: str) -> str:
    return f"pydocs_{clave}"


t_insercion: dict[tuple[str, str], float] = {}

for clave in MODELOS:
    nombre = nombre_coleccion(clave)
    vectores = vectores_chunks[clave]

    # Qdrant
    if qdrant.collection_exists(nombre):
        qdrant.delete_collection(nombre)
    qdrant.create_collection(
        nombre, vectors_config=VectorParams(size=dimensiones[clave], distance=Distance.COSINE)
    )
    inicio = time.perf_counter()
    for i in range(0, len(chunks), LOTE_INSERCION):
        qdrant.upsert(nombre, points=[
            PointStruct(id=c["id"], vector=v, payload={"fuente": c["fuente"], "texto": c["texto"]})
            for c, v in zip(chunks[i:i + LOTE_INSERCION], vectores[i:i + LOTE_INSERCION])
        ])
    t_insercion[(clave, "Qdrant")] = time.perf_counter() - inicio

    # Chroma
    if nombre in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(nombre)
    coleccion = chroma.create_collection(nombre, configuration={"hnsw": {"space": "cosine"}})
    inicio = time.perf_counter()
    for i in range(0, len(chunks), LOTE_INSERCION):
        lote = chunks[i:i + LOTE_INSERCION]
        coleccion.add(
            ids=[str(c["id"]) for c in lote],
            embeddings=vectores[i:i + LOTE_INSERCION],
            documents=[c["texto"] for c in lote],
            metadatas=[{"fuente": c["fuente"]} for c in lote],
        )
    t_insercion[(clave, "Chroma")] = time.perf_counter() - inicio

for (clave, base), t in t_insercion.items():
    print(f"{nombre_coleccion(clave)} en {base}: {t:.1f} s")

## 7. Preguntas del golden set

Se leen de las tablas de `docs/GOLDEN_SET_PYTHON_DOCS.md`; ninguna pregunta está escrita en el código. Antes de evaluar se avisa, con el ID y la ruta, de las fuentes que no existen en el corpus. La evidencia literal de cada pregunta A-D se lee de `docs/golden_set_evidencias.json`, y se comprueba que cada subcadena cabe entera en algún chunk de su fuente (si no, esa pregunta no podría acertar nunca).

In [ ]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

evidencias = leer_evidencias(RUTA_EVIDENCIAS)
print("Evidencias partidas por el troceo:", evidencias_fuera_de_chunks(evidencias, chunks) or "ninguna")

## 8. Evaluación con k=4

Para cada colección y cada pregunta se recuperan 4 resultados y se mide la latencia (embedding de la query más búsqueda). Se calculan dos rangos: `rango` (acierta si la `fuente` del chunk es una de las fuentes de la pregunta) y `rango_evidencia` (además, el texto del chunk contiene la evidencia de esa fuente). En las preguntas E (sin fuentes) se guarda el score del mejor resultado como similitud coseno (en Chroma, `1 - distancia`).

In [ ]:
def buscar_qdrant(nombre: str, vector: list[float]) -> tuple[list[dict], float]:
    res = qdrant.query_points(nombre, query=vector, limit=K, with_payload=True).points
    return [{"fuente": p.payload["fuente"], "texto": p.payload["texto"]} for p in res], res[0].score


def buscar_chroma(nombre: str, vector: list[float]) -> tuple[list[dict], float]:
    res = chroma.get_collection(nombre).query(query_embeddings=[vector], n_results=K)
    recuperados = [
        {"fuente": m["fuente"], "texto": t}
        for m, t in zip(res["metadatas"][0], res["documents"][0])
    ]
    return recuperados, 1 - res["distances"][0][0]


BUSCADORES = {"Qdrant": buscar_qdrant, "Chroma": buscar_chroma}

resultados = []
for clave in MODELOS:
    for base, buscar in BUSCADORES.items():
        for p in preguntas:
            inicio = time.perf_counter()
            vector = embeber(clave, [preparar_query(clave, p["query"])])[0]
            recuperados, mejor_score = buscar(nombre_coleccion(clave), vector)
            latencia_ms = (time.perf_counter() - inicio) * 1000
            resultados.append({
                "modelo": MODELOS[clave]["nombre"], "clave": clave, "base": base,
                "id": p["id"], "categoria": p["categoria"],
                "rango": rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"]) if p["fuentes"] else None,
                "rango_evidencia": rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]]) if p["fuentes"] else None,
                "con_fuentes": bool(p["fuentes"]),
                "score": mejor_score, "latencia_ms": latencia_ms,
            })

df = pd.DataFrame(resultados)
print(len(df), "evaluaciones")

## 9. Métricas por colección (preguntas A-D)

hit@1, hit@4 y MRR@4 sobre las preguntas con fuentes, con los dos criterios: por fuente y por evidencia (más estricto).

In [ ]:
def metricas(grupo: pd.DataFrame) -> pd.Series:
    """Métricas por fuente y por evidencia (prefijo `ev_`)."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        filas.update({prefijo + n: v for n, v in calcular_metricas(rangos, K).items()})
    return pd.Series(filas)


con_fuentes = df[df["con_fuentes"]]
por_coleccion = con_fuentes.groupby(["modelo", "base"]).apply(metricas, include_groups=False)
por_coleccion

## 10. Desglose por categoría (A-D)

In [ ]:
por_categoria = con_fuentes.groupby(["categoria", "modelo", "base"]).apply(metricas, include_groups=False)
por_categoria

## 11. Preguntas E: score del mejor resultado

Las preguntas E no tienen respuesta en el corpus. Un score alto en el mejor resultado indica que el modelo «inventa» relevancia.

In [ ]:
df[~df["con_fuentes"]].pivot_table(index="id", columns=["modelo", "base"], values="score").round(3)

## 12. Tabla resumen (modelo × base)

La dimensión sale del embedding calculado. El tiempo de indexación es embeddings del modelo más inserción en esa base.

In [ ]:
filas = []
for clave, cfg in MODELOS.items():
    for base in BUSCADORES:
        m = por_coleccion.loc[(cfg["nombre"], base)]
        sub = df[(df["clave"] == clave) & (df["base"] == base)]
        filas.append({
            "modelo": cfg["nombre"], "base": base,
            "hit@1": m["hit@1"], f"hit@{K}": m[f"hit@{K}"], "MRR": m["mrr"],
            "ev_hit@1": m["ev_hit@1"], f"ev_hit@{K}": m[f"ev_hit@{K}"], "ev_MRR": m["ev_mrr"],
            "latencia media (ms)": sub["latencia_ms"].mean(),
            "indexación (s)": t_embeddings[clave] + t_insercion[(clave, base)],
            "dimensión": dimensiones[clave],
        })
resumen = pd.DataFrame(filas)
resumen.round(3)